## Bibliotecas

In [2]:
import requests
import json
import pandas as pd
from time import sleep
from datetime import datetime, timedelta
from IPython.display import HTML, display
import time
import re
import csv

# Importando tabelas do Google Sheets

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## API CRM

In [3]:
# Pega os dados do CRM por meio da API

def query_appointments(current_page,start_date,end_date):

  token = "145418|arQc09gsrcSNJipgDRaM4Ep6rl3aJGkLtDMnxa0u"
  endpoint = "https://open-api.eprocorpo.com.br/graphql"
  headers = {'Content-Type': 'application/json',
             "authorization": f"Bearer {token}"}

  query = """query ($filters: AppointmentFiltersInput, $pagination: PaginationInput) {
                    fetchAppointments(filters: $filters, pagination: $pagination) {
                        meta {
                            lastPage
                        }
                        data {
                            id
                            status {
                                label
                            }
                            store {
                                name
                            }
                            customer {
                                id
                                name
                                telephones {
                                    number
                                }
                            }
                            procedure{
                                name
                            }
                            startDate
                        }
                    }
                }"""

  variables = {
                "filters": {
                    "startDateRange": {
                        "start": start_date,
                        "end": end_date,
                    },
                },
                "pagination": {
                    "currentPage": current_page,
                    "perPage": 1000,
                },
            }

  response = requests.post(endpoint, json={"query": query,"variables":variables}, headers=headers)

  if response.status_code == 200:

    response = response.json()
    total_pages = response["data"]["fetchAppointments"]["meta"]["lastPage"]

    print(f"\rQuerying - {current_page}/{total_pages} pages",end="")

    progress_index = current_page//(total_pages/100)
    out.update(progress(progress_index, 100))

    return response

  else:
    print(response)
    return "error"



def progress(value, max=100):
    return HTML("""
        <progress
            value='{value}'
            max='{max}',
            style='width: 80%'
        >
            {value}
        </progress>
    """.format(value=value, max=max))

### Update Appointments File

In [4]:
# Cria a base com os agendamentos dos últimos 30 dias e dos próximos 30 dias

days_to_offset = 30 # Seleciona a quantidade de dias antes e depois
d_minus_30 = (datetime.today() - timedelta(days=days_to_offset)).strftime('%Y-%m-%d')
d_plus_30 = (datetime.today() + timedelta(days=days_to_offset)).strftime('%Y-%m-%d')

def create_appointmens(file_name,start_date,end_date):
  # Faz as requisições e salva o resultado em um arquivo CSV
  folder_path = "/content/drive/MyDrive/_whatsapp-project/appointments_30d"

  save_path = f"{folder_path}/{file_name}"

  current_page = 1

  api_response = query_appointments(current_page,start_date,end_date)

  if (api_response == "error"):
    print("Something went wrong with the query")
    print("Waiting a bit...")
    sleep(10)
    print("Trying Again!")
    api_response = query_lead(current_page,start_date,end_date)

    if api_response == "error":
      print("Error with the query... aborting")

  api_data = api_response["data"]["fetchAppointments"]["data"]
  api_data_length = len(api_data)


  results_list = [["appointments_id","status_label","store_name","customer_id","customer_name","telephones","procedure_name","startDate"]]

  while (api_data_length > 0):

    for data_row in api_data:

      appointments_id = data_row["id"]

      status_data = data_row["status"]
      status_label = status_data["label"]

      try:
        store_data = data_row["store"]
        store_name = store_data["name"]
      except:
        store_name = ""

      customer_data = data_row["customer"]
      customer_name = customer_data["name"]
      customer_id = customer_data["id"]

      try:
        customer_telephones = customer_data["telephones"][0]
        telephone = customer_telephones["number"]
      except:
        telephone = ""


      procedure_data = data_row["procedure"]
      procedure_name = procedure_data["name"]

      startDate = data_row["startDate"]

      results_row = [appointments_id,status_label,store_name,customer_id,customer_name,telephone,procedure_name,startDate]

      results_list.append(results_row)

    current_page += 1

    api_response = query_appointments(current_page,start_date,end_date)

    if (api_response == "error"):
      print("Something went wrong with the query")
      print("Waiting a bit...")
      sleep(10)
      print("Trying Again!")
      api_response = query_lead(current_page,start_date,end_date)

      if api_response == "error":
        print("Error with the query... aborting")
        break

    api_data = api_response["data"]["fetchAppointments"]["data"]
    api_data_length = len(api_data)

  df = pd.DataFrame(results_list[1:],columns = results_list[0])
  df.to_csv(save_path, index=False)
  print('\r' + ' ' * 30, end='')
  print('\r', end='')
  print(f"Query Concluida com sucesso! - {start_date} - {end_date}")

out = display(progress(0, 100), display_id=True)
create_appointmens("appointments_30d.csv",d_minus_30,d_plus_30)

Query Concluida com sucesso! - 2024-10-22 - 2024-12-21


## API SocialHub

In [5]:
# Envias as mensagens por meio da API do Social Hub

def send_message(telephone, message):
    url = "https://apinew.socialhub.pro/api/sendMessage"

    telephone = str(telephone)

    # ID Botox rmvYoOnWD5WjcH7Bx5lYTZkGMX2vweN1
    # ID Ativo de Falta 688gNxW8Ygqn2Zrb7k9Vy1BYBN9mtf5b
    request_data = {
        "api_token": "688gNxW8Ygqn2Zrb7k9Vy1BYBN9mtf5b",  # ID da API no Social HUB BOTOX
        "phone": telephone,
        "message": message,
        "preview_url": True
    }

    headers = {
        "Content-Type": "application/json"
    }

    response = requests.post(url, headers=headers, data=json.dumps(request_data))

    if response.status_code != 200:
        print(f"Request failed with status code: {response.status_code}")
        print(f"Response content: {response.text}")
        return None

    data = response.json()

    return data

def send_message_with_file(telephone, message,file_path):
    url = "https://apinew.socialhub.pro/api/sendMessage"

    telephone = str(telephone)

    request_data = {
        "api_token": "688gNxW8Ygqn2Zrb7k9Vy1BYBN9mtf5b",  # ID da API no Social HUB BOTOX
        "phone": telephone,
        "message": message,
        "preview_url": True
    }

    headers = {
        "Content-Type": "application/json"
    }

    with open(file_path, 'rb') as file:
        files = {'file': file}
        response = requests.post(url, data=request_data, files=files)

    if response.status_code != 200:
        print(f"Request failed with status code: {response.status_code}")
        print(f"Response content: {response.text}")
        return None

    data = response.json()

    return data

def enviar_mensagem(telefone,mensagem,file_name):

  if file_name == None:
    data = send_message(telefone,mensagem)
  else:
    data = send_message_with_file(telefone,mensagem,file_name)

  return data

In [6]:
import requests
from time import sleep

def send_message(telephone, message):
    url = "https://apinew.socialhub.pro/api/sendMessage"
    telephone = str(telephone)
    request_data = {
        "api_token": "688gNxW8Ygqn2Zrb7k9Vy1BYBN9mtf5b",  # Exemplo de ID da API
        "phone": telephone,
        "message": message,
        "preview_url": True
    }
    headers = {"Content-Type": "application/json"}

    for attempt in range(10):  # Tenta até 3 vezes
        try:
            response = requests.post(url, headers=headers, json=request_data, timeout=(10, 30))  # 10s connect, 30s read
            if response.status_code == 200:
                return response.json()
            else:
                print(f"Request failed with status code: {response.status_code}")
                print(f"Response content: {response.text}")
        except requests.exceptions.Timeout:
            print("Timeout occurred. Retrying...")
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {str(e)}")
        sleep(10)  # Espera 10 segundos antes de tentar novamente

    return {"success": False}  # Retorna False se falhar todas as tentativas

# Adapte também a função para enviar mensagens com arquivos
def send_message_with_file(telephone, message, file_path):
    url = "https://apinew.socialhub.pro/api/sendMessage"
    telephone = str(telephone)
    request_data = {
        "api_token": "688gNxW8Ygqn2Zrb7k9Vy1BYBN9mtf5b",
        "phone": telephone,
        "message": message,
        "preview_url": True
    }
    headers = {"Content-Type": "application/json"}

    for attempt in range(10):  # Tenta até 3 vezes
        try:
            with open(file_path, 'rb') as file:
                files = {'file': file}
                response = requests.post(url, data=request_data, files=files, timeout=(10, 30))
            if response.status_code == 200:
                return response.json()
            else:
                print(f"Request failed with status code: {response.status_code}")
                print(f"Response content: {response.text}")
        except requests.exceptions.Timeout:
            print("Timeout occurred. Retrying...")
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {str(e)}")
        sleep(10)  # Espera 10 segundos antes de tentar novamente

    return None  # Retorna None se falhar todas as tentativas

# Use `enviar_mensagem` para chamar as funções acima.
def enviar_mensagem(telefone, mensagem, file_name=None):
    if file_name is None:
        return send_message(telefone, mensagem)
    else:
        return send_message_with_file(telefone, mensagem, file_name)

## Carrega Bases

In [7]:
# Função para atualizar CSV de Mensagens

def update_csv_mensagens(dados_mensagens):
  with open(mensagens_log_file, 'a', newline='') as csvfile:
    writer = csv.writer(csvfile, delimiter=',')
    writer.writerow(dados_mensagens)

In [8]:
# Load databases

appointents_file = "/content/drive/MyDrive/_whatsapp-project/appointments_30d/appointments_30d.csv"
mensagens_log_file = "/content/drive/MyDrive/_whatsapp-project/sent_messages_absences.csv"

appointents_df = pd.read_csv(appointents_file) # Relatório de agendamentos do CRM
base_mensagens_df = pd.read_csv(mensagens_log_file) # Log de mensagens enviadas

In [9]:
# Filtra log de mensagens excluindo mensagens enviadas há mais de 30 dias

base_mensagens_df['data_de_envio'] = pd.to_datetime(base_mensagens_df['data_de_envio'])
date_30_days_ago = datetime.now() - timedelta(days=30)
base_mensagens_df = base_mensagens_df[base_mensagens_df['data_de_envio'] >= date_30_days_ago]

# Groupby do log de mensagens

contar_mensagens = base_mensagens_df.groupby(["telefone","status_fluxo"]).agg({"telefone":"count"})
contar_mensagens = contar_mensagens.rename(columns={"telefone": "mensagens_count"})

contar_mensagens = contar_mensagens.reset_index()
contar_mensagens["telefone"] = contar_mensagens["telefone"].astype(str)

In [10]:
import re

# Função para limpar telefones
def clean_telephone(telefone):
    telefone = str(telefone).split('.')[0]  # Remove qualquer coisa após o ponto decimal
    telefone = re.sub(r'[^\d]', '', telefone)  # Remove todos os caracteres não numéricos
    if telefone.startswith('55'):
        telefone = telefone[2:]  # Remove o prefixo +55
    return telefone

# Arruma os telefones
appointents_df["telephones"] = appointents_df["telephones"].astype(str)
appointents_df["telephones"] = appointents_df["telephones"].apply(clean_telephone)

## Filtrando base e enviando mensagens

In [11]:
# Cria a base filtrada para enviar as mensagens

# Cria a coluna com a diferença de datas
appointents_df['startDate'] = pd.to_datetime(appointents_df['startDate'])
appointents_df['startDate'] = appointents_df['startDate'].dt.normalize()
today = datetime.today()
appointents_df['dias_do_agendamento'] = (today - appointents_df['startDate']).dt.days

procedimento_regex = r".*AVALIAÇÃO INJETÁVEIS E INVASIVOS.*|.*AVALIAÇÃO ESTÉTICA.*"
unidades_regex_exclusiva = r".*HOMA.*|.*PLÁSTICA.*"
agendado_regex = r".*Agendado.*|.*Reagendado.*|.*Atendido.*"
list_of_stores = ['TIJUCA', 'TUCURUVI', 'MOOCA', 'COPACABANA', 'MOEMA', 'LAPA',
                  'IPIRANGA', 'SANTOS', 'OSASCO', 'SOROCABA','LONDRINA', 'JARDINS', 'CAMPINAS',
                  'TATUAPÉ', 'ALPHAVILLE', 'ITAIM', 'SANTO AMARO', 'RIBEIRÃO PRETO']

# Usamos quando o volume era mais reduzido!
# unidade_regex_inclusiva = r".*Ipiranga.*|.*Itaim.*|.*Jardins.*|.*Sorocaba.*|.*Campinas.*|.*Tucuruvi.*|.*Lapa.*|.*Moema.*|.*Tatuapé.*" # Adicionar unidades que queremos ver no relatório
# unidade_incluir_mask = appointents_df['store_name'].str.contains(unidade_regex_inclusiva, na=False, case=False) # Filtra somente unidades selecionadas

procedimento_mask = appointents_df['procedure_name'].str.contains(procedimento_regex, na=False, case=False) #Filtra somente procedimentos válidos
unidade_excluir_mask = ~appointents_df['store_name'].str.contains(unidades_regex_exclusiva, na=False, case=False) # Filtra tudo que não é Home ou Plástica
unidade_incluir_mask = appointents_df['store_name'].isin(list_of_stores)

appointents_df["eh_agendamento"] =  appointents_df["status_label"].str.contains(agendado_regex, na=False, case=False) # Cria coluna flag de agendamento

# Filtra base de agendamentos
filtered_df = appointents_df.loc[procedimento_mask & unidade_excluir_mask & unidade_incluir_mask]

# Acrescenta coluna com a quantidade de mensagens por status (cancelado, falta)
filtered_df = filtered_df.merge(contar_mensagens, how='left', left_on=["telephones","status_label"], right_on=["telefone","status_fluxo"])
filtered_df.drop(columns=["telefone","status_fluxo"],inplace=True)
filtered_df.fillna(0,inplace=True)

# Adiciona coluna com a flag que diz se o cliente tem algum agendamento
filtered_df["tem agendamento"] = filtered_df.groupby(by=["telephones"])["eh_agendamento"].transform("max")

# Filtra telefones
filtered_df = filtered_df.loc[filtered_df["tem agendamento"] == False] # Não tem agendamentos
filtered_df = filtered_df.loc[filtered_df["dias_do_agendamento"] > 0] # Registros anteriores a hoje

# Cria base telefones únicos para enviar mensagem

df_para_mandar_mensagens = filtered_df.groupby(by=["telephones","status_label"]).agg({"dias_do_agendamento":"min",'mensagens_count':'max',"store_name":"first","customer_name":"first","startDate":"max"}).reset_index()

status_atendido_regex = r".*Atendido.*|.*Confirmado.*"
status_atendido_mask = ~df_para_mandar_mensagens['status_label'].str.contains(status_atendido_regex, na=False, case=False) # Remove Atendio e Confirmado

df_para_mandar_mensagens = df_para_mandar_mensagens.loc[status_atendido_mask]
# Pega telefones únicos com a data mais recente
df_para_mandar_mensagens = df_para_mandar_mensagens.loc[df_para_mandar_mensagens.groupby('telephones')['dias_do_agendamento'].idxmin()]

In [12]:
# nova_linha_df = pd.DataFrame([{
#     'telephones': '11963546222',
#     'status_label': 'Falta',
#     'dias_do_agendamento': 5,
#     'mensagens_count': 0,
#     'store_name': 'IPIRANGA',
#     'customer_name': 'LUIS',
#     'startDate': '2024-07-18 00:00:00'
# }])

# # Adicionar a nova linha usando pd.concat
# df_para_mandar_mensagens = pd.concat([df_para_mandar_mensagens, nova_linha_df], ignore_index=True)
# df_para_mandar_mensagens

# # Delete data from dataframe
# df_para_mandar_mensagens = df_para_mandar_mensagens[df_para_mandar_mensagens['telephones'] == '11963546222']
# df_para_mandar_mensagens

In [13]:
# Pega o texto e outros dados das mensagens

url_da_tabela_de_mensagens = 'https://docs.google.com/spreadsheets/d/1iNt8-2qmQKYD0McDDrUUb9QLBn5G2h6M5yUa8oo2isg/edit'
ss = gc.open_by_url(url_da_tabela_de_mensagens)
sheet = ss.worksheet("Mensagens_ES")
mensagens = sheet.get_all_records()

# Transforma os dados em um dicionário para hash indexing
mensagens_dic = {}
for item in mensagens:
  tipo_de_mensagem = item["nome_mensagem"]
  mensagens_dic[tipo_de_mensagem] = item

In [14]:
df_para_mandar_mensagens['mensagens_count'] = df_para_mandar_mensagens['mensagens_count'].astype(int)
# df_para_mandar_mensagens['dias_do_agendamento'] = df_para_mandar_mensagens['dias_do_agendamento'].astype(int)

df_para_mandar_mensagens

,telephones,status_label,dias_do_agendamento,mensagens_count,store_name,customer_name,startDate
0,0,Cancelado,21,3,JARDINS,Gabriela Leite,2024-10-31
1,1122310121,Cancelado,28,0,JARDINS,Dalila Gomes Soares,2024-10-24
2,1134093340,Falta,28,3,TATUAPÉ,Wanda Barbosa Lemes,2024-10-24
3,1141949982,Falta,26,3,SANTO AMARO,Cecilia da Cruz Machado,2024-10-26
4,1145537707,Cancelado,2,0,OSASCO,Ana Mirian Ribeiro Pires da Silva,2024-11-19
...,...,...,...,...,...,...,...
2935,92994539205,Falta,7,3,COPACABANA,Vera Gomes,2024-11-14
2936,96991734203,Falta,27,3,JARDINS,Janayna Kelly,2024-10-25
2937,988731422,Cancelado,30,3,LAPA,Patricia E Costa,2024-10-22
2938,991542654,Cancelado,2,0,LONDRINA,Elizabeth Zanin de Oliveira,2024-11-19


In [15]:
# # Criando o DataFrame com uma linha de dados personalizados
# data = {
#     'telephones': ['5511963546222'],  # Coloque seu número aqui no formato correto
#     'status_label': ['Falta'],  # Personalize o status
#     'dias_do_agendamento': [4],  # Dias até o agendamento
#     'mensagens_count': [0.0],  # Contagem de mensagens
#     'store_name': ['TATUAPÉ'],  # Nome da loja
#     'customer_name': ['Luis Faria'],  # Nome do cliente
#     'startDate': ['2024-08-14']  # Data do início
# }

# df_para_mandar_mensagens = pd.DataFrame(data)
# df_para_mandar_mensagens

## Enviar mensagem

In [16]:
# Substitui as variáveis no texto da mensagem

def ajusta_mensagem(row,texto_mensagem):

  store_name = row["store_name"].capitalize()
  customer_name = row["customer_name"].split()[0].capitalize()

  startDate = row["startDate"]
  startDate = startDate.strftime('%Y-%m-%d')
  startDate = datetime.strptime(startDate, '%Y-%m-%d').strftime('%d/%m/%Y')

  texto_mensagem = texto_mensagem.replace("[nome]", customer_name)
  texto_mensagem = texto_mensagem.replace("[unidade]", store_name)
  texto_mensagem = texto_mensagem.replace("[data]", startDate)

  return texto_mensagem

In [17]:
import time

# Total time to sleep
total_time = 1 * 60 * 90  #+ (15 * 60) # 1 hour + 15m
interval = 60
display_interval = 1  # countdown
# Loop that sleeps until time to go
for remaining_time in range(int(total_time), 0, -interval):

    time.sleep(interval)
    if remaining_time % (display_interval * 60) == 0:
        remaining_minutes = remaining_time // 60
        print(f"Time left: {remaining_minutes} minutes")

print("Finished waiting.. ready for next")

Time left: 90 minutes
Time left: 89 minutes
Time left: 88 minutes
Time left: 87 minutes
Time left: 86 minutes
Time left: 85 minutes
Time left: 84 minutes
Time left: 83 minutes
Time left: 82 minutes
Time left: 81 minutes
Time left: 80 minutes
Time left: 79 minutes
Time left: 78 minutes
Time left: 77 minutes
Time left: 76 minutes
Time left: 75 minutes
Time left: 74 minutes
Time left: 73 minutes
Time left: 72 minutes
Time left: 71 minutes
Time left: 70 minutes
Time left: 69 minutes
Time left: 68 minutes
Time left: 67 minutes
Time left: 66 minutes
Time left: 65 minutes
Time left: 64 minutes
Time left: 63 minutes
Time left: 62 minutes
Time left: 61 minutes
Time left: 60 minutes
Time left: 59 minutes
Time left: 58 minutes
Time left: 57 minutes
Time left: 56 minutes
Time left: 55 minutes
Time left: 54 minutes
Time left: 53 minutes
Time left: 52 minutes
Time left: 51 minutes
Time left: 50 minutes
Time left: 49 minutes
Time left: 48 minutes
Time left: 47 minutes
Time left: 46 minutes
Time left:

In [18]:
from time import sleep
data_de_envio = datetime.today().strftime('%Y-%m-%d')

for index, row in df_para_mandar_mensagens.iterrows():
    telephone = row["telephones"]
    dias_do_agendamento = row["dias_do_agendamento"]
    mensagens_count = row["mensagens_count"]
    status_label = row["status_label"]

    if status_label == "Falta":
        if (dias_do_agendamento >= 1) & (dias_do_agendamento < 7):
            if (mensagens_count == 0):
                nome_mensagem = "Mensagem 1"
                mensagem = mensagens_dic[nome_mensagem]
                texto_mensagem = mensagem["texto_mensagem"]
                texto_mensagem = ajusta_mensagem(row, texto_mensagem)

                file_id = mensagem["arquivo"]
                if file_id == '':
                    file_id = None
                response = enviar_mensagem(telephone, texto_mensagem, file_id)
                if response["success"] == True:
                    update_csv_mensagens([telephone, status_label, nome_mensagem, data_de_envio])
                    print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
                else:
                    print(f"erro {response}")
                # Espera 20 segundos após o envio
                sleep(20)

        elif (dias_do_agendamento >= 7) & (dias_do_agendamento < 14):
            if (mensagens_count == 1):
                nome_mensagem = "Mensagem 2"
                mensagem = mensagens_dic[nome_mensagem]
                texto_mensagem = mensagem["texto_mensagem"]
                texto_mensagem = ajusta_mensagem(row, texto_mensagem)

                file_id = mensagem["arquivo"]
                if file_id == '':
                    file_id = None
                response = enviar_mensagem(telephone, texto_mensagem, file_id)
                if response["success"] == True:
                    update_csv_mensagens([telephone, status_label, nome_mensagem, data_de_envio])
                    print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
                else:
                    print(f"erro {response}")
                # Espera 20 segundos após o envio
                sleep(20)

        elif (dias_do_agendamento >= 14):
            if (mensagens_count == 2):
                nome_mensagem = "Mensagem 3"
                mensagem = mensagens_dic[nome_mensagem]
                texto_mensagem = mensagem["texto_mensagem"]
                texto_mensagem = ajusta_mensagem(row, texto_mensagem)

                file_id = mensagem["arquivo"]
                if file_id == '':
                    file_id = None
                response = enviar_mensagem(telephone, texto_mensagem, file_id)
                if response["success"] == True:
                    update_csv_mensagens([telephone, status_label, nome_mensagem, data_de_envio])
                    print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
                else:
                    print(f"erro {response}")
                # Espera 20 segundos após o envio
                sleep(20)

    elif status_label == "Cancelado":
        if (dias_do_agendamento >= 1) & (dias_do_agendamento < 7):
            if (mensagens_count == 0):
                nome_mensagem = "Mensagem 4"
                mensagem = mensagens_dic[nome_mensagem]
                texto_mensagem = mensagem["texto_mensagem"]
                texto_mensagem = ajusta_mensagem(row, texto_mensagem)

                file_id = mensagem["arquivo"]
                if file_id == '':
                    file_id = None
                response = enviar_mensagem(telephone, texto_mensagem, file_id)
                if response["success"] == True:
                    update_csv_mensagens([telephone, status_label, nome_mensagem, data_de_envio])
                    print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
                else:
                    print(f"erro {response}")
                # Espera 20 segundos após o envio
                sleep(20)

        elif (dias_do_agendamento >= 7) & (dias_do_agendamento < 14):
            if (mensagens_count == 1):
                nome_mensagem = "Mensagem 5"
                mensagem = mensagens_dic[nome_mensagem]
                texto_mensagem = mensagem["texto_mensagem"]
                texto_mensagem = ajusta_mensagem(row, texto_mensagem)

                file_id = mensagem["arquivo"]
                if file_id == '':
                    file_id = None
                response = enviar_mensagem(telephone, texto_mensagem, file_id)

                if response["success"] == True:
                    update_csv_mensagens([telephone, status_label, nome_mensagem, data_de_envio])
                    print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
                else:
                    print(f"erro {response}")
                # Espera 20 segundos após o envio
                sleep(20)

        elif (dias_do_agendamento >= 14):
            if (mensagens_count == 2):
                nome_mensagem = "Mensagem 6"
                mensagem = mensagens_dic[nome_mensagem]
                texto_mensagem = mensagem["texto_mensagem"]
                texto_mensagem = ajusta_mensagem(row, texto_mensagem)

                file_id = mensagem["arquivo"]
                if file_id == '':
                    file_id = None
                response = enviar_mensagem(telephone, texto_mensagem, file_id)
                if response["success"] == True:
                    update_csv_mensagens([telephone, status_label, nome_mensagem, data_de_envio])
                    print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
                else:
                    print(f"erro {response}")
                # Espera 20 segundos após o envio
                sleep(20)

Mensagem 4 enviada com sucesso para: 1145537707
Mensagem 2 enviada com sucesso para: 1158311429
Mensagem 1 enviada com sucesso para: 11910175501
Mensagem 1 enviada com sucesso para: 11910185026
Mensagem 5 enviada com sucesso para: 11910325010
Mensagem 5 enviada com sucesso para: 11910637969
Mensagem 1 enviada com sucesso para: 11911138068
Mensagem 2 enviada com sucesso para: 11911300113
Mensagem 1 enviada com sucesso para: 11911798905
Mensagem 2 enviada com sucesso para: 11912656757
Mensagem 2 enviada com sucesso para: 11913026007
Mensagem 4 enviada com sucesso para: 11913058558
Mensagem 2 enviada com sucesso para: 11913360723
Mensagem 6 enviada com sucesso para: 11913603836
Mensagem 4 enviada com sucesso para: 11914888145
Mensagem 5 enviada com sucesso para: 11914923391
Mensagem 2 enviada com sucesso para: 11915056721
Mensagem 5 enviada com sucesso para: 11915063342
Mensagem 2 enviada com sucesso para: 11915063383
Mensagem 2 enviada com sucesso para: 11916187593
Mensagem 1 enviada com